# Predicción binaria del income (>$50K anual)

En este cuaderno, vamos a entrenar diversos modelos sobre el dataset **folktables**, buscando predecir si el salario anual de una persona es mayor que $50K. Se va a prestar especial atención a la disparidad que surge respecto a la variable raza (o grupo étnico, para mayor exactitud).

# 1. Preparación de los datos

El primer paso es descargar los datos del dataset. Vamos a tomar datos de 2018 para siete estados de EEUU:

In [21]:
from folktables import ACSDataSource, ACSIncome

# Acotamos los datos del estudio (p. ej. año)
data_source = ACSDataSource(survey_year=2018,
                            horizon='1-Year',
                            survey='person')

# Descargamos para 6 estados
acs_data = data_source.get_data(states=['AL', 'AK', 'AZ', 'NH', 'ME', 'SD', 'CA', 'NY', 'TX'], download=True)


Llevamos los datos a formato Numpy y visualizamos:

In [22]:
import pandas as pd
import numpy as np

# Obtenemos los datos en formato Numpy
X, y, group = ACSIncome.df_to_numpy(acs_data)

# Lo metemos en un dataframe para ver los datos
df = pd.DataFrame(X, columns=ACSIncome.features)
df['protected_group'] = group # en este caso, el grupo protegido sale como RAC1P en los features
df['label'] = y
print(df.head(6).to_string(index=False))

 AGEP  COW  SCHL  MAR   OCCP  POBP  RELP  WKHP  SEX  RAC1P  protected_group  label
 18.0  1.0  18.0  5.0 4720.0  13.0  17.0  21.0  2.0    2.0                2  False
 53.0  5.0  17.0  5.0 3605.0  18.0  16.0  40.0  1.0    1.0                1  False
 41.0  1.0  16.0  5.0 7330.0   1.0  17.0  40.0  1.0    1.0                1  False
 18.0  6.0  18.0  5.0 2722.0   1.0  17.0   2.0  2.0    1.0                1  False
 21.0  5.0  19.0  5.0 3870.0  12.0  17.0  50.0  1.0    1.0                1  False
 37.0  5.0  16.0  4.0 9620.0   1.0  16.0  35.0  1.0    2.0                2  False


Hacemos el split train/test de los datos:

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(X, y, group,
                                                                     test_size=0.2,
                                                                     random_state=42,
                                                                     stratify=y)

# Los escalamos
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


## 2. Preparar el modelo sin meta-learning

Vamos a entrenar dos modelos distintos:
- Uno va a ver la variable raza explícitamente en los datos de entrenamiento.
- El otro no.

Ambos van a ser este (simple) modelo de Pytorch:

In [24]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
device = "cuda" if torch.cuda.is_available() else "cpu"

class ModeloEnBruto(nn.Module):
    def __init__(self, tam_entrada): # tam_entrada es 9 si no hay raza, 10 si la raza se añade
        super().__init__()
        self.modelo = nn.Sequential(
                nn.Linear(tam_entrada, 64),
                nn.ReLU(),
                nn.Linear(64, 128),
                nn.ReLU(),
                nn.Linear(128,64),
                nn.ReLU(),
                nn.Linear(64, 1))
    def forward(self, x):
        return self.modelo(x).squeeze(-1)


## 3. Entrenar ambos modelos: viendo o no la raza en el entrenamiento

### 3.1. Entrenar con la raza como feature explícito

Instanciamos el modelo, pasamos los datos a tensores, y creamos los dataloaders:

In [25]:
device = "cpu"
modelo = ModeloEnBruto(10).to(device)
X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32, device=device)
y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
y_test_t = torch.tensor(y_test,  dtype=torch.float32, device=device)
g_train_t = torch.tensor(g_train, dtype=torch.long,  device=device)
g_test_t = torch.tensor(g_test,  dtype=torch.long,  device=device)

# DataLoaders
batch_size = 1024

train_ds = TensorDataset(X_train_t, y_train_t, g_train_t)
test_ds = TensorDataset(X_test_t, y_test_t, g_test_t)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(modelo.parameters(), lr=1e-3)



Entrenamos el modelo:

In [26]:
# Loop de entrenamiento
def train_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total = 0
    for Xb, yb, gb in loader:
        Xb = Xb.to(device)
        yb = yb.float().to(device)

        optimizer.zero_grad()
        logits = model(Xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * Xb.size(0)
        total += Xb.size(0)
    return total_loss / total

def eval_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for Xb, yb, gb in loader:
            Xb = Xb.to(device)
            yb = yb.to(device)
            logits = model(Xb)
            preds = (torch.sigmoid(logits) >= 0.5).long()
            correct += (preds == yb).sum().item()
            total += yb.numel()
    return correct / total

num_epochs = 15

acc0 = eval_accuracy(modelo, test_loader)
print(f"Before training | test_acc={acc0:.4f}")

for epoch in range(num_epochs):
    train_loss = train_epoch(modelo, train_loader)
    acc = eval_accuracy(modelo, test_loader)

    if (epoch%10==4):
       print(f"Epoch {epoch+1}/{num_epochs} | loss={train_loss:.4f} | test_acc={acc:.4f}")


Before training | test_acc=0.6036
Epoch 5/15 | loss=0.4302 | test_acc=0.7913
Epoch 15/15 | loss=0.4203 | test_acc=0.7967


### 3.2. Entrenar el modelo SIN la raza

Para entrenar el modelo sin ver la raza, primero tenemos que volver a preparar los datos: Eliminar la columna RAC1P, escalar los datos, llevar a tensores, reinstanciar el modelo...

In [27]:
X_no_race = np.delete(X, 9, axis=1)

# PREPARAR LOS DATOS
X_train2, X_test2, y_train2, y_test2, g_train2, g_test2 = train_test_split(
    X_no_race, y, group,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler2 = StandardScaler()
X_train2 = scaler2.fit_transform(X_train2)
X_test2  = scaler2.transform(X_test2)

X_train2_t = torch.tensor(X_train2, dtype=torch.float32, device=device)
X_test2_t  = torch.tensor(X_test2,  dtype=torch.float32, device=device)
y_train2_t = torch.tensor(y_train2, dtype=torch.float32, device=device)
y_test2_t  = torch.tensor(y_test2,  dtype=torch.float32, device=device)
g_test2_t  = torch.tensor(g_test2,   dtype=torch.long,   device=device)

# INSTANCIAR Y ENTRENAR EL MODELO
modelo2 = ModeloEnBruto(9).to(device)
optimizer2 = torch.optim.Adam(modelo2.parameters(), lr=1e-3)

train2_ds = TensorDataset(X_train2_t, y_train2_t)
test2_ds  = TensorDataset(X_test2_t,  y_test2_t, g_test2_t)

train2_loader = DataLoader(train2_ds, batch_size=1024, shuffle=True)
test2_loader  = DataLoader(test2_ds,  batch_size=1024, shuffle=False)

def train_epoch2(model, loader):
    model.train()
    total_loss = 0
    total = 0
    for Xb, yb in loader:
        Xb = Xb.to(device)
        yb = yb.to(device)

        optimizer2.zero_grad()
        logits = model(Xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer2.step()

        total_loss += loss.item() * Xb.size(0)
        total += Xb.size(0)
    return total_loss / total

def eval_accuracy2(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for Xb, yb, _ in loader:
            Xb = Xb.to(device)
            yb = yb.to(device)
            logits = model(Xb)
            preds = (torch.sigmoid(logits) >= 0.5).long()
            correct += (preds == yb).sum().item()
            total += yb.numel()
    return correct / total

for epoch in range(15):
    _ = train_epoch2(modelo2, train2_loader)
    if (epoch % 10 == 4):
        acc = eval_accuracy2(modelo2, test2_loader)
        print(f"Epoch {epoch+1}/15 | test_acc={acc:.4f}")


Epoch 5/15 | test_acc=0.7923
Epoch 15/15 | test_acc=0.7982


## 4. Comparando resultados: ¿ayuda ver la raza?

Primero, necesitamos algo de código auxiliar para obtener las métricas desglosadas por raza:

In [28]:
def eval_full_metrics_by_group(model, X_test_t, y_test_t, g_test_t):
    model.eval()
    with torch.no_grad():
        logits = model(X_test_t)
        preds = (torch.sigmoid(logits) >= 0.5).long()

    groups = g_test_t.cpu().numpy()
    labels = y_test_t.cpu().numpy().astype(int)
    preds_np = preds.cpu().numpy()

    results = {}

    for g in np.unique(groups):
        mask = (groups == g)

        y_true = labels[mask]
        y_pred = preds_np[mask]

        n = len(y_true)

        # Confusion matrix terms
        TP = np.sum((y_pred == 1) & (y_true == 1))
        FP = np.sum((y_pred == 1) & (y_true == 0))
        TN = np.sum((y_pred == 0) & (y_true == 0))
        FN = np.sum((y_pred == 0) & (y_true == 1))

        accuracy = (TP + TN) / n if n > 0 else float('nan')

        # Avoid division by zero
        TPR = TP / (TP + FN) if (TP + FN) > 0 else 0.0   # sensitivity / recall
        FPR = FP / (FP + TN) if (FP + TN) > 0 else 0.0

        results[g] = {
            "n_members": n,
            "TP": TP,
            "FP": FP,
            "TN": TN,
            "FN": FN,
            "accuracy": accuracy,
            "TPR": TPR,
            "FPR": FPR
        }

    return results

def eval_accuracy_by_group(model, X_test_t, y_test_t, g_test_t):
    model.eval()
    with torch.no_grad():
        logits = model(X_test_t)
        preds = (torch.sigmoid(logits) >= 0.5).long()

    results = {}
    groups = g_test_t.cpu().numpy()
    labels = y_test_t.cpu().numpy()
    preds_np = preds.cpu().numpy()

    for g in np.unique(groups):
        mask = (groups == g)
        acc = (preds_np[mask] == labels[mask]).mean()
        results[g] = acc

    return results

metrics_by_group = eval_full_metrics_by_group(modelo, X_test_t, y_test_t, g_test_t)
metrics_by_group2 = eval_full_metrics_by_group(modelo2, X_test2_t, y_test2_t, g_test2_t)


### 4.1. Resultados del modelo que ha visto la raza como dato

In [29]:
print("Modelo entrenado con la raza como dato explícito.")

df_metrics = pd.DataFrame.from_dict(metrics_by_group, orient="index")
df_metrics.index.name = "Race"


df_metrics = df_metrics[[
    "n_members", "accuracy", "TP", "FP", "TN", "FN", "TPR", "FPR"
]]

print(df_metrics.to_string())
print("="*70)

Modelo entrenado con la raza como dato explícito.
      n_members  accuracy     TP    FP     TN    FN       TPR       FPR
Race                                                                   
1         72789  0.791095  22762  7905  34821  7301  0.757143  0.185016
2          7899  0.790353   1359   688   4884   968  0.584014  0.123475
3           863  0.818076    100    58    606    99  0.502513  0.087349
4           186  0.849462     19     5    139    23  0.452381  0.034722
5           220  0.822727     14    18    167    21  0.400000  0.097297
6         10086  0.802499   3701  1039   4393   953  0.795230  0.191274
7           168  0.815476     32    13    105    18  0.640000  0.110169
8          7415  0.842751    592   348   5657   818  0.419858  0.057952
9          3088  0.804080    661   271   1822   334  0.664322  0.129479


### 4.2. Resultados del modelo que NO ha visto la raza como dato

In [30]:
print("Modelo entrenado sin la raza como dato explícito.")

df_metrics2 = pd.DataFrame.from_dict(metrics_by_group2, orient="index")
df_metrics2.index.name = "Race"

df_metrics2 = df_metrics2[[
    "n_members", "accuracy", "TP", "FP", "TN", "FN", "TPR", "FPR"
]]

print(df_metrics2.to_string())


Modelo entrenado sin la raza como dato explícito.
      n_members  accuracy     TP    FP     TN    FN       TPR       FPR
Race                                                                   
1         72789  0.793087  22455  7453  35273  7608  0.746931  0.174437
2          7899  0.786429   1466   826   4746   861  0.629996  0.148241
3           863  0.808806    123    89    575    76  0.618090  0.134036
4           186  0.827957     20    10    134    22  0.476190  0.069444
5           220  0.827273     15    18    167    20  0.428571  0.097297
6         10086  0.800714   3617   973   4459  1037  0.777181  0.179124
7           168  0.821429     31    11    107    19  0.620000  0.093220
8          7415  0.845988    620   352   5653   790  0.439716  0.058618
9          3088  0.816710    686   257   1836   309  0.689447  0.122790


### 4.3. Conclusiones de estos resultados

Lo primero y más evidente: **existe un problema de unfairness**, pues ninguna de las definiciones de fairness se respeta. En este caso, es de esperar que haya disparate impact, pues la raza condiciona socialmente muchos aspectos directamente relacionados con el salario. En este caso, es especialmente relevante la diferencia en *equal opportunity*: como patrón general, las razas más representadas tienen un mayor TPR, hecho más evidente en el modelo que aprende explícitamente esta variable. Sin embargo, esta es difícilmente la única razón para la diferencia: hay que tener en cuenta varias variables (los estados elegidos para la muestra y su situación económica, la actitud social general hacia cada raza...). Además, con muestras tan pequeñas (p.ej. 18 muestras en la séptima, nativos hawaiianos), difícilmente se puede establecer una explicación causal de los valores obtenidos.

De todas formas, este caso es ideal para probar el enfoque de meta-learning.  En esta situación, un approach que nos garantice mini-max fairness (dado que en algunos casos, como el 4, el TPR es inaceptable) sería ideal.

## 5. Atacando el problema con MAML

**Nota:** Quiero entender bien y rehacer este código por mi cuenta. Para implementar el meta-learning, he tirado bastante de influencia GPTil.

### 5.1. Código auxiliar

Primero, necesitamos varias cosas, como:
- La lista de razas (race_groups) para dividir el entrenamiento en tasks.
- Funciones auxiliares para el sampling de datos en el proceso de meta-learning, para realizar un fine-tune en cada grupo, para mostrar los datos...

In [31]:
race_groups = np.unique(g_train2)

def sample_task_batches(group_id, k_support=128, k_query=128):
    idxs = np.where(g_train2 == group_id)[0]
    replace = len(idxs) < (k_support + k_query)
    chosen = np.random.choice(idxs, size=k_support + k_query, replace=replace)
    support_idx = chosen[:k_support]
    query_idx = chosen[k_support:]

    support_x = X_train2_t[support_idx]
    support_y = y_train2_t[support_idx]
    query_x = X_train2_t[query_idx]
    query_y = y_train2_t[query_idx]
    return support_x, support_y, query_x, query_y

def fine_tune_for_group(group_id, steps=5, lr=0.03):
    # === SUPPORT SET ===
    support_idx = np.where(g_train2 == group_id)[0]
    X_support = X_train2_t[support_idx]
    y_support = y_train2_t[support_idx]

    adapted = ModeloEnBruto(9).to(device)
    adapted.load_state_dict(meta_model.state_dict())

    opt = torch.optim.SGD(adapted.parameters(), lr=lr)

    # Fine-tuning
    for _ in range(steps):
        opt.zero_grad()
        logits = adapted(X_support)
        loss = loss_fn(logits, y_support)
        loss.backward()
        opt.step()

    # Evaluar en el grupo
    adapted.eval()
    with torch.no_grad():
        mask = (g_test2_t == group_id)
        X_eval = X_test2_t[mask]
        y_eval = y_test2_t[mask]

        n = X_eval.shape[0]
        if n == 0:
            return {
                "n_members": 0,
                "accuracy": float('nan'),
                "TP": 0, "FP": 0, "TN": 0, "FN": 0,
                "TPR": float('nan'), "FPR": float('nan')
            }

        logits = adapted(X_eval)
        preds = (torch.sigmoid(logits) >= 0.5).long()

        y_true = y_eval.cpu().numpy().astype(int)
        y_pred = preds.cpu().numpy().astype(int)

        # Métricas
        TP = ((y_pred == 1) & (y_true == 1)).sum()
        FP = ((y_pred == 1) & (y_true == 0)).sum()
        TN = ((y_pred == 0) & (y_true == 0)).sum()
        FN = ((y_pred == 0) & (y_true == 1)).sum()

        accuracy = (TP + TN) / n
        TPR = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        FPR = FP / (FP + TN) if (FP + TN) > 0 else 0.0

    return {
        "n_members": n,
        "accuracy": accuracy,
        "TP": TP,
        "FP": FP,
        "TN": TN,
        "FN": FN,
        "TPR": TPR,
        "FPR": FPR
    }



### 5.2. Redefinir el modelo

Tenemos que redefinir el modelo para poder usar por separado el meta-modelo y los modelos entrenados en el inner loop, sin mezclar los parámetros:

In [32]:
import torch.nn.functional as F

class ModeloEnBruto(nn.Module):
    def __init__(self, tam_entrada):  # tam_entrada es 9 si no hay raza, 10 si la raza se añade
        super().__init__()
        self.fc1 = nn.Linear(tam_entrada, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 1)

    def forward(self, x, params=None):
        if params is None:
            x = F.relu(self.fc1(x))
            x = F.relu(self.fc2(x))
            x = self.fc3(x)
        else:
            x = F.linear(x, params['fc1.weight'], params['fc1.bias'])
            x = F.relu(x)
            x = F.linear(x, params['fc2.weight'], params['fc2.bias'])
            x = F.relu(x)
            x = F.linear(x, params['fc3.weight'], params['fc3.bias'])
        return x.squeeze(-1)


Definimos el meta-modelo, y ejecutamos un loop de entrenamiento:

In [33]:
meta_model = ModeloEnBruto(9).to(device)
meta_optimizer = torch.optim.Adam(meta_model.parameters(), lr=1e-3)
inner_lr = 0.05
inner_steps = 5
meta_epochs = 40

for epoch in range(meta_epochs):
    meta_optimizer.zero_grad()
    meta_loss = 0.0 # loss total de todas las razas

    # para cada raza
    for g_id in race_groups:
        # samplear un batch
        support_x, support_y, query_x, query_y = sample_task_batches(g_id)

        params = {name: param for name, param in meta_model.named_parameters()}

        # entrenar sobre el batch inner_step veces
        for _ in range(inner_steps):
            support_logits = meta_model(support_x, params)
            support_loss = loss_fn(support_logits, support_y)
            grads = torch.autograd.grad(support_loss, params.values(), create_graph=True)
            params = {
                name: param - inner_lr * grad
                for (name, param), grad in zip(params.items(), grads)
            }

        query_logits = meta_model(query_x, params)
        task_loss = loss_fn(query_logits, query_y)
        meta_loss = meta_loss + task_loss

    meta_loss = meta_loss / len(race_groups)
    meta_loss.backward()
    meta_optimizer.step()

    if (epoch + 1) % 2 == 0:
        print(f"Meta-epoch {epoch+1}/{meta_epochs} | meta-loss={meta_loss.item():.4f}")



Meta-epoch 2/40 | meta-loss=0.6000
Meta-epoch 4/40 | meta-loss=0.5786
Meta-epoch 6/40 | meta-loss=0.5446
Meta-epoch 8/40 | meta-loss=0.5304
Meta-epoch 10/40 | meta-loss=0.5012
Meta-epoch 12/40 | meta-loss=0.5139
Meta-epoch 14/40 | meta-loss=0.5049
Meta-epoch 16/40 | meta-loss=0.4785
Meta-epoch 18/40 | meta-loss=0.4748
Meta-epoch 20/40 | meta-loss=0.4582
Meta-epoch 22/40 | meta-loss=0.4574
Meta-epoch 24/40 | meta-loss=0.4527
Meta-epoch 26/40 | meta-loss=0.4825
Meta-epoch 28/40 | meta-loss=0.4649
Meta-epoch 30/40 | meta-loss=0.4486
Meta-epoch 32/40 | meta-loss=0.4470
Meta-epoch 34/40 | meta-loss=0.4425
Meta-epoch 36/40 | meta-loss=0.4798
Meta-epoch 38/40 | meta-loss=0.4463
Meta-epoch 40/40 | meta-loss=0.4310


### 5.3. Fine-tunear y probar en cada raza

In [34]:
maml_results = {}
for g_id in race_groups:
    maml_results[g_id] = fine_tune_for_group(g_id)
df_maml = pd.DataFrame.from_dict(maml_results, orient="index")
df_maml.index.name = "Race"
print(df_maml.to_string())

      n_members  accuracy     TP    FP     TN    FN       TPR       FPR
Race                                                                   
1         72789  0.775529  21386  7662  35064  8677  0.711373  0.179329
2          7899  0.771363   1260   739   4833  1067  0.541470  0.132627
3           863  0.806489    104    72    592    95  0.522613  0.108434
4           186  0.822581     17     8    136    25  0.404762  0.055556
5           220  0.809091     12    19    166    23  0.342857  0.102703
6         10086  0.771069   3482  1137   4295  1172  0.748174  0.209315
7           168  0.809524     30    12    106    20  0.600000  0.101695
8          7415  0.837357    520   316   5689   890  0.368794  0.052623
9          3088  0.792746    620   265   1828   375  0.623116  0.126613


Como bien podemos observar, en este caso, no ha generado una diferencia significativa con los resultados anteriores:

In [35]:
print(df_metrics2.to_string())


      n_members  accuracy     TP    FP     TN    FN       TPR       FPR
Race                                                                   
1         72789  0.793087  22455  7453  35273  7608  0.746931  0.174437
2          7899  0.786429   1466   826   4746   861  0.629996  0.148241
3           863  0.808806    123    89    575    76  0.618090  0.134036
4           186  0.827957     20    10    134    22  0.476190  0.069444
5           220  0.827273     15    18    167    20  0.428571  0.097297
6         10086  0.800714   3617   973   4459  1037  0.777181  0.179124
7           168  0.821429     31    11    107    19  0.620000  0.093220
8          7415  0.845988    620   352   5653   790  0.439716  0.058618
9          3088  0.816710    686   257   1836   309  0.689447  0.122790
